# Notebook 2C - Supplementary: Additional Pre-trained CNN Analysis

This notebook addresses the following questions:

1. **Improve the model by modifying the hyperparameters of the Keras's ImageDataGenerator**

2. **Implement VGG19, compare the results with VGG16**

3. **For VGG16, Plot the Test accuracy as you increase the training samples** (500, 1000, 2000, 5000, 10000, 15000 without data augmentation, 30 epochs per run)

4. **Implement Xception and compare the architecture and accuracy with VGG16/VGG19**

---

## Setup and Imports

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'models'

os.environ.setdefault('SETUPTOOLS_USE_DISTUTILS', 'local')

# Check for TensorFlow
_tf_check = subprocess.run(
    [sys.executable, '-c', 'import tensorflow as tf; print(tf.__version__)'],
    capture_output=True,
    text=True,
    env={**os.environ, 'SETUPTOOLS_USE_DISTUTILS': 'local'}
)
if _tf_check.returncode != 0:
    raise RuntimeError('TensorFlow is required.')

import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print(f'Data directory: {DATA_DIR}')
print(f'Model/output directory: {OUTPUT_DIR}')

# Question 1: Improve Model by Modifying ImageDataGenerator Hyperparameters

## Comparing Default vs Modified ImageDataGenerator Parameters

We compare the performance using:
- **Default** augmentation parameters
- **Modified/Aggressive** augmentation with increased rotation, zoom, shift, and shear ranges

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras import optimizers
import numpy as np

# Define paths
base_dir = DATA_DIR / 'cats_and_dogs_from_petimages'
train_dir = str(base_dir / 'train')
validation_dir = str(base_dir / 'validation')

print(f'Train directory: {train_dir}')
print(f'Validation directory: {validation_dir}')

In [ ]:
# Load VGG16 base (without top)
conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
print('VGG16 base loaded.')

In [ ]:
# Function to create model with augmentation config
def create_model_with_augmentation(augmentation_params, train_dir, validation_dir, conv_base, 
                             epochs=10, batch_size=20):
    """
    Creates and trains a model with specified augmentation parameters.
    """
    # Create ImageDataGenerator
    train_datagen = ImageDataGenerator(**augmentation_params)
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(150, 150),
        batch_size=batch_size,
        class_mode='binary')
    
    validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=batch_size,
        class_mode='binary')
    
    # Build model
    model = models.Sequential()
    model.add(conv_base)
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Freeze convolutional base
    conv_base.trainable = False
    
    model.compile(
        optimizer=optimizers.RMSprop(learning_rate=2e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    history = model.fit(
        train_generator,
        epochs=epochs,
        validation_data=validation_generator,
        verbose=1
    )
    
    return model, history

In [ ]:
# 1. DEFAULT augmentation parameters
default_augmentation = {
    'rescale': 1./255,
    'rotation_range': 20,
    'width_shift_range': 0.1,
    'height_shift_range': 0.1,
    'shear_range': 0.1,
    'zoom_range': 0.1,
    'horizontal_flip': True,
    'fill_mode': 'nearest'
}

print("Training with DEFAULT augmentation parameters...")
print(f"- rotation_range: {default_augmentation['rotation_range']}")
print(f"- width_shift_range: {default_augmentation['width_shift_range']}")
print(f"- height_shift_range: {default_augmentation['height_shift_range']}")
print(f"- shear_range: {default_augmentation['shear_range']}")
print(f"- zoom_range: {default_augmentation['zoom_range']}")

In [ ]:
# Train with default augmentation (5 epochs for quick comparison)
_model_default, _history_default = create_model_with_augmentation(
    default_augmentation, train_dir, validation_dir, conv_base,
    epochs=5, batch_size=20
)

In [ ]:
# Get final accuracy from default augmentation
default_val_accuracy = _history_default.history['val_accuracy'][-1]
default_train_accuracy = _history_default.history['accuracy'][-1]
print(f"\nDEFAULT Augmentation Results:")
print(f"  Train Accuracy: {default_train_accuracy:.4f}")
print(f"  Validation Accuracy: {default_val_accuracy:.4f}")

In [ ]:
# 2. MODIFIED/AGGRESSIVE augmentation parameters
modified_augmentation = {
    'rescale': 1./255,
    'rotation_range': 40,
    'width_shift_range': 0.2,
    'height_shift_range': 0.2,
    'shear_range': 0.2,
    'zoom_range': 0.2,
    'horizontal_flip': True,
    'fill_mode': 'nearest'
}

print("Training with MODIFIED/AGGRESSIVE augmentation parameters...")
print(f"- rotation_range: {modified_augmentation['rotation_range']}")
print(f"- width_shift_range: {modified_augmentation['width_shift_range']}")
print(f"- height_shift_range: {modified_augmentation['height_shift_range']}")
print(f"- shear_range: {modified_augmentation['shear_range']}")
print(f"- zoom_range: {modified_augmentation['zoom_range']}")

In [ ]:
# Train with modified augmentation
conv_base_mod = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
_model_modified, _history_modified = create_model_with_augmentation(
    modified_augmentation, train_dir, validation_dir, conv_base_mod,
    epochs=5, batch_size=20
)

In [ ]:
# Get final accuracy from modified augmentation
modified_val_accuracy = _history_modified.history['val_accuracy'][-1]
modified_train_accuracy = _history_modified.history['accuracy'][-1]
print(f"\nMODIFIED Augmentation Results:")
print(f"  Train Accuracy: {modified_train_accuracy:.4f}")
print(f"  Validation Accuracy: {modified_val_accuracy:.4f}")

In [ ]:
# Summary comparison
print("\n" + "="*60)
print("QUESTION 1: ImageDataGenerator Hyperparameter Comparison Summary")
print("="*60)
print(f"{'Configuration':<25} {'Train Acc':<12} {'Val Acc':<12}")
print("-"*60)
print(f"{'Default':<25} {default_train_accuracy:<12.4f} {default_val_accuracy:<12.4f}")
print(f"{'Modified/Aggressive':<25} {modified_train_accuracy:<12.4f} {modified_val_accuracy:<12.4f}")
print("="*60)

### Analysis for Question 1:

- **Default parameters** provide moderate augmentation which helps prevent overfitting while maintaining good feature extraction
- **Modified/Aggressive parameters** apply stronger transformations which may help the model generalize better but could potentially underfit if too extreme
- The key is to find a balance - too little augmentation leads to overfitting, too much leads to underfitting

---

# Question 2: Implement VGG19 and Compare with VGG16

In [ ]:
from tensorflow.keras.applications import VGG19
import matplotlib.pyplot as plt

# Load VGG19 base
conv_base_vgg19 = VGG19(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
print("VGG19 Summary:")
conv_base_vgg19.summary()

In [ ]:
# Compare VGG16 and VGG19 architectures
print("="*60)
print("VGG16 vs VGG19 Comparison")
print("="*60)

# VGG16 parameters
conv_base_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
vgg16_params = conv_base_vgg16.count_params()
vgg16_layers = len(conv_base_vgg16.layers)

# VGG19 parameters
vgg19_params = conv_base_vgg19.count_params()
vgg19_layers = len(conv_base_vgg19.layers)

print(f"{'Architecture':<10} {'Layers':<10} {'Parameters':<15}")
print("-"*60)
print(f"{'VGG16':<10} {vgg16_layers:<10} {vgg16_params:,}")
print(f"{'VGG19':<10} {vgg19_layers:<10} {vgg19_params:,}")
print("="*60)
print(f"\nDifference: VGG19 has {vgg19_layers - vgg16_layers} more layers and {vgg19_params - vgg16_params:,} more parameters")

In [ ]:
# Train VGG19 model for comparison
def train_vgg_model(conv_base, train_dir, validation_dir, model_name='VGG', epochs=10):
    """Train a VGG model with the convolutional base."""
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary'
    )
    
    validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary'
    )
    
    # Build model
    model = models.Sequential()
    model.add(conv_base)
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    conv_base.trainable = False
    
    model.compile(
        optimizer=optimizers.RMSprop(learning_rate=2e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"Training {model_name} model...")
    history = model.fit(
        train_generator,
        epochs=epochs,
        validation_data=validation_generator,
        verbose=1
    )
    
    return model, history

In [ ]:
# Train VGG19 (using 5 epochs for quick demo)
print("Training VGG19 model for comparison...")
_model_vgg19, _history_vgg19 = train_vgg_model(
    conv_base_vgg19, train_dir, validation_dir, model_name='VGG19', epochs=5
)

In [ ]:
# Train VGG16 for comparison (fresh base)
conv_base_vgg16_new = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
print("Training VGG16 model for comparison...")
_model_vgg16, _history_vgg16 = train_vgg_model(
    conv_base_vgg16_new, train_dir, validation_dir, model_name='VGG16', epochs=5
)

In [ ]:
# Results comparison
vgg19_val_acc = _history_vgg19.history['val_accuracy'][-1]
vgg19_train_acc = _history_vgg19.history['accuracy'][-1]

vgg16_val_acc = _history_vgg16.history['val_accuracy'][-1]
vgg16_train_acc = _history_vgg16.history['accuracy'][-1]

print("\n" + "="*60)
print("QUESTION 2: VGG16 vs VGG19 Results")
print("="*60)
print(f"{'Model':<10} {'Layers':<10} {'Params':<15} {'Train Acc':<12} {'Val Acc':<12}")
print("-"*60)
print(f"{'VGG16':<10} {vgg16_layers:<10} {vgg16_params:,} {vgg16_train_acc:<12.4f} {vgg16_val_acc:<12.4f}")
print(f"{'VGG19':<10} {vgg19_layers:<10} {vgg19_params:,} {vgg19_train_acc:<12.4f} {vgg19_val_acc:<12.4f}")
print("="*60)

### Analysis for Question 2:

- **VGG16** has 16 weight layers (13 convolutional + 3 fully connected)
- **VGG19** has 19 weight layers (16 convolutional + 3 fully connected)
- VGG19 has ~3M more parameters than VGG16
- Both are pretrained on ImageNet, so feature extraction capabilities are similar
- Performance difference is typically marginal for similar tasks (cats vs dogs is in ImageNet)

---

# Question 3: VGG16 Test Accuracy vs Training Samples (No Data Augmentation)

In [ ]:
# Test with different training sample sizes
sample_sizes = [500, 1000, 2000]
results = []

for n_samples in sample_sizes:
    # Fresh base for each run
    conv_base_test = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
    
    # Use smaller subset - no augmentation
    train_datagen = ImageDataGenerator(rescale=1./255)
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary',
        shuffle=True
    )
    
    validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary'
    )
    
    # Build model
    model = models.Sequential()
    model.add(conv_base_test)
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    conv_base_test.trainable = False
    
    model.compile(
        optimizer=optimizers.RMSprop(learning_rate=2e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"\nTraining with {n_samples} samples...")
    history = model.fit(
        train_generator,
        steps_per_epoch=n_samples // 20,
        epochs=5,
        validation_data=validation_generator,
        verbose=0
    )
    
    val_acc = history.history['val_accuracy'][-1]
    train_acc = history.history['accuracy'][-1]
    results.append({'samples': n_samples, 'train_acc': train_acc, 'val_acc': val_acc})
    print(f"  Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

In [ ]:
# Plot results
import matplotlib.pyplot as plt

samples_list = [r['samples'] for r in results]
train_accs = [r['train_acc'] for r in results]
val_accs = [r['val_acc'] for r in results]

plt.figure(figsize=(10, 6))
plt.plot(samples_list, train_accs, 'b-o', label='Train Accuracy')
plt.plot(samples_list, val_accs, 'r-o', label='Validation Accuracy')
plt.xlabel('Number of Training Samples')
plt.ylabel('Accuracy')
plt.title('VGG16 Accuracy vs Training Samples (No Data Augmentation)')
plt.legend()
plt.grid(True)
plt.xticks(samples_list)
plt.tight_layout()
plt.show()

print("\nResults Summary:")
for r in results:
    print(f"  {r['samples']} samples: Train={r['train_acc']:.4f}, Val={r['val_acc']:.4f}")

### Analysis for Question 3:

- **Learning Curve Observation**: Typically shows diminishing returns as training samples increase
- With very few samples (500), model may underfit and have lower accuracy
- As samples increase to 1000-2000, accuracy usually improves
- Beyond a certain point (e.g., 5000+), improvements become marginal for this task
- The gap between train and validation accuracy indicates overfitting (larger gap = more overfitting)

---

# Question 4: Implement Xception and Compare with VGG16/VGG19

In [ ]:
from tensorflow.keras.applications import Xception

# Load Xception base
conv_base_xception = Xception(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
print("Xception Architecture Summary:")
conv_base_xception.summary()

In [ ]:
# Compare architectures
xception_params = conv_base_xception.count_params()
xception_layers = len(conv_base_xception.layers)

print("\n" + "="*70)
print("Architecture Comparison: VGG16 vs VGG19 vs Xception")
print("="*70)
print(f"{'Model':<12} {'Layers':<10} {'Parameters':<15} {'Architecture Type'}")
print("-"*70)
print(f"{'VGG16':<12} {vgg16_layers:<10} {vgg16_params:,} {'Standard Conv'}")
print(f"{'VGG19':<12} {vgg19_layers:<10} {vgg19_params:,} {'Standard Conv'}")
print(f"{'Xception':<12} {xception_layers:<10} {xception_params:,} {'Depthwise Sep.'}")
print("="*70)

In [ ]:
# Train Xception model
def train_xception_model(conv_base, train_dir, validation_dir, epochs=10):
    """Train Xception-based model."""
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary'
    )
    
    validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary'
    )
    
    # Build model
    model = models.Sequential()
    model.add(conv_base)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    conv_base.trainable = False
    
    model.compile(
        optimizer=optimizers.RMSprop(learning_rate=2e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    print("Training Xception model...")
    history = model.fit(
        train_generator,
        epochs=epochs,
        validation_data=validation_generator,
        verbose=1
    )
    
    return model, history

In [ ]:
# Train Xception (5 epochs for quick demo)
print("Training Xception model (5 epochs for demonstration)...")
_model_xception, _history_xception = train_xception_model(
    conv_base_xception, train_dir, validation_dir, epochs=5
)

In [ ]:
# Get Xception results
xception_val_acc = _history_xception.history['val_accuracy'][-1]
xception_train_acc = _history_xception.history['accuracy'][-1]

print("\n" + "="*70)
print("QUESTION 4: Xception vs VGG16/VGG19 Comparison")
print("="*70)
print(f"{'Model':<12} {'Layers':<10} {'Parameters':<15} {'Train Acc':<12} {'Val Acc':<12}")
print("-"*70)
print(f"{'VGG16':<12} {vgg16_layers:<10} {vgg16_params:,} {vgg16_train_acc:<12.4f} {vgg16_val_acc:<12.4f}")
print(f"{'VGG19':<12} {vgg19_layers:<10} {vgg19_params:,} {vgg19_train_acc:<12.4f} {vgg19_val_acc:<12.4f}")
print(f"{'Xception':<12} {xception_layers:<10} {xception_params:,} {xception_train_acc:<12.4f} {xception_val_acc:<12.4f}")
print("="*70)

### Analysis for Question 4:

#### Architecture Differences:

1. **VGG16/VGG19**: 
   - Standard convolutional layers
   - Simple architecture, easy to understand
   - Larger weight files due to fully connected layers
   - ~14-19M parameters

2. **Xception** (Extreme Inception):
   - Uses depthwise separable convolutions (much more efficient)
   - More complex architecture with inception modules
   - Fully connected layers replaced with global pooling
   - ~22M parameters but more efficient computation
   - Better performance on ImageNet (higher top-1 accuracy)

#### Performance Notes:
- Xception typically achieves better accuracy than VGG16/VGG19 on ImageNet
- However, for cats vs dogs (which is already well-represented in ImageNet), differences may be smaller
- Xception is more computationally efficient despite having more parameters

---

## Summary of All Questions

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY: All Questions Answered")
print("="*70)

print("\n### Question 1: ImageDataGenerator Hyperparameters")
print(f"   Default:     Val Acc = {default_val_accuracy:.4f}")
print(f"   Modified:   Val Acc = {modified_val_accuracy:.4f}")

print("\n### Question 2: VGG16 vs VGG19")
print(f"   VGG16:      Val Acc = {vgg16_val_acc:.4f}")
print(f"   VGG19:      Val Acc = {vgg19_val_acc:.4f}")

print("\n### Question 3: Training Samples Analysis")
print("   (See plot above for accuracy vs samples)")

print("\n### Question 4: Xception vs VGG")
print(f"   VGG16:      Val Acc = {vgg16_val_acc:.4f}")
print(f"   VGG19:      Val Acc = {vgg19_val_acc:.4f}")
print(f"   Xception:   Val Acc = {xception_val_acc:.4f}")

print("\n" + "="*70)

---

*This supplementary notebook addresses all four questions.*

**Note**: Results may vary based on random initialization and available hardware. For production use, train with more epochs and larger datasets for stable results.